# Phase 6 (Abstractive Summarization) — Run #4: **Knowledge distillation** (Enc-Dec + Houlsby teacher → 7-block Dec-03 student)

The mandatory KD experiment. Edit **Cell 2** only. Same data pipeline, chunk pairs, decoding sweep (BERTScore selection), evaluation and submission as runs #1–#3. **One full run gives the final submission.**

### Teacher and student
* **Teacher = run #2, Enc-Dec 01 + Houlsby adapters** (63.5M parameters), the strongest Phase-6 run on VAL: R1 0.596 / R2 0.336 / RL 0.309 / **BERTScore 0.856**. It beats the TF-IDF extractive baseline on every metric, and run #1 (Dec-02 + LoRMA: 0.569 / 0.304 / 0.287 / 0.851).
* **Student = Phase-1 Decoder-03 (ALiBi + MLA) with every second block removed**: blocks 0, 2, …, 12 → 7 blocks, ~36.6M parameters (58 % of the base), as in Phase 5. The student is a *different architecture* (decoder-only, causal) and *smaller* than the teacher (asserted). It is fully fine-tuned, because removing 6 blocks breaks the residual stream it must re-learn.

### Two distillation signals (Hinton et al. 2015; Kim & Rush 2016), both exported by run #2
1. **Word-level KD** on the gold targets. For every target token the teacher stored its top-k logits and the log-sum-exp over the whole vocabulary, so `p_k = exp(z_k − lse)` are its **true** probabilities of those k tokens, and `r = 1 − Σ p_k` is the mass on all other tokens. The student is matched on this partition `{top-k tokens, rest}`:

   `KL(p ‖ q) = Σ_k p_k log(p_k / q_k) + r log(r / q_rest)`,  `q_rest = 1 − Σ_k q_k`

   This is the *exact* KL divergence between the two distributions coarsened to k + 1 outcomes (≥ 0, and 0 if and only if the student matches the teacher on every top-k token and on the rest). Per target token: `loss = (1 − α)·CE(gold) + α·KL`. T = 1 because the exported log-sum-exp is at T = 1 (Kim & Rush also use T = 1 for sequence models).
   Cross-architecture word-level KD is only possible because **all Phase-1 models share one tokenizer**, so the teacher's and the student's output distributions live on the same 32k-token vocabulary, token for token.
2. **Sequence-level KD**: the teacher's own greedy chunk summaries (~15k training chunks) are added as extra targets. Kim & Rush show that the student learns the teacher's *mode*, which is easier to fit than the multi-modal gold data, and that this matters most for small students.

The targets must be token-identical to the teacher's, so the student uses the teacher's chunking, target cap (`TGT_MAX = 255`) and alignment threshold; Cell 12 asserts this against the teacher file and checks each pair's length before using its logits.

**Control run** (for the report's "does KD help?"): `KD_ALPHA = 0` and `SEQ_KD = False` trains the same student on gold targets only.

### Run it
1. Upload run #2's `teacher_p6_encdec_adapter.pt` as a Kaggle dataset and set `TEACHER_EXPORT` (Cell 2).
2. `SMOKE_TEST=True` (~10 min), then the full run. Paste the **"PASTE TO CLAUDE"** block.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle, copy, hashlib, bisect, subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p6_kd_dec03_student"
METHOD     = "kd"             # knowledge distillation; the student is fully fine-tuned

# ---- Distillation ---------------------------------------------------------------------------------
KD_ALPHA = 0.5                # weight of the word-level KD term per gold target token; 0.0 = no word-level KD
SEQ_KD   = True               # add the teacher's generated chunk summaries as extra training targets
TEACHER_EXPORT = Path("/kaggle/input/datasets/ascyrax/caps-p6-teacher/teacher_p6_encdec_adapter.pt") if IS_KAGGLE else Path("./input/teacher.pt")

# ---- Student = Phase-1 Decoder-03 checkpoint with some blocks removed --------------------------------
STUDENT_KEEP_LAYERS = [0, 2, 4, 6, 8, 10, 12]      # alternate blocks of the 13-block model -> 7 blocks
FREEZE_EMBEDDINGS   = False
SMOKE_TEST = False            # True: ~150 steps on a small subset, a few VAL documents generated, NO submission / export

# ---- Phase-1 checkpoint of the student (caps_decoder_only_architecture_03.ipynb) -------------------------------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-03-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data / split (audit) ------------------------------------------------------------------------
HF_REPO        = "Exploration-Lab/CS-781-Capstone"     # Phase_6_Summ/{Train/train.json, Test/test.json}
LOCAL_DATA_DIR = None
P6             = "Phase_6_Summ"
VAL_N, VAL_SALT, VAL_CHECKSUM_EXPECTED = 200, "p6-val-v1", "5720e140c147"
MIN_SUMM_WORDS, MAX_SUMM_WORDS = 20, 5000       # training pairs only (VAL is kept as in the audit)
MAX_FULL_CHARS = 150_000                        # longer documents (< 1 %) -> first 2/3 + last 1/3 of this many characters

# ---- Chunk pairs ---------------------------------------------------------------------------------
CHUNK_TOKENS = 512           # document tokens per chunk
TGT_MAX      = 255           # = the teacher's (asserted): word-level KD needs token-identical targets
MIN_ALIGN_F1 = 0.2           # weaker alignments are not assigned to a chunk (audit: 1.9 % of summary sentences)
EMPTY_KEEP   = 0.5           # share of chunks WITHOUT aligned summary text kept as EOS-only examples (resampled every epoch)
TAG_TEMPLATE = "[part {i}/{n}] "
SEP_TEXT     = "\n\nSUMMARY:\n"
N_PROC       = 4

# ---- Training ------------------------------------------------------------------------------------
EPOCHS        = 4            # 7 blocks: cheaper per step than run #1; ~25 % more sequences with SEQ_KD (check the >> speed line)
MICRO_TOKENS  = 12288        # padded tokens per micro-batch (n_seqs x longest); lower it on OOM
STEP_SEQS     = 32           # sequences per optimiser step (gradient accumulation)
WARMUP_FRAC   = 0.03
GRAD_CLIP     = 1.0
GRAD_CKPT     = False
TIME_BUDGET_MIN = 400        # training stops after this (best epoch kept); leaves ~3 h for evaluation, test and export
HP = {"kd": dict(lr=3e-4, wd=0.0)}   # full fine-tuning of the truncated student
SELECT_BY          = "loss"  # "loss" (VAL teacher-forced loss) | "rouge" (quick ROUGE on VAL_GEN_DOCS_EPOCH docs)
VAL_GEN_DOCS_EPOCH = 40

# ---- Generation ----------------------------------------------------------------------------------
GEN = dict(max_chunks=16,        # audit: 16 x 512 reads 95.7 % of all tokens
           max_new=TGT_MAX, min_new_first=32,   # the first chunk must say something (no empty summaries)
           min_new_all=64,                      # [EOS] banned for this many tokens in EVERY chunk (run #1 sweep: 0 -> 79 % empty chunks)
           eos_threshold=0.0,                   # > 0: stop only if p(EOS) >= this (greedy otherwise stops whenever EOS is the single top token)
           no_repeat_ngram=4, rep_penalty=1.1, batch=64,
           len_factor=1.25, dedup=True,
           trim="proportional")                 # over budget: "proportional" (every chunk keeps the same share) | "tail" (cut the end)
# Each entry overrides GEN for one sweep run (run #1: min_new_all 0 / 32 / 64 -> R1 0.200 / 0.531 / 0.586, still rising at 64)
DECODE_SWEEP = [dict(min_new_all=64), dict(min_new_all=96), dict(min_new_all=128),
                dict(min_new_all=32, eos_threshold=0.5), dict(min_new_all=64, eos_threshold=0.5),
                dict(min_new_all=96, trim="tail"), dict(min_new_all=96, max_chunks=24), dict(min_new_all=96, len_factor=1.5)]
SWEEP_DOCS   = 100           # VAL docs used to pick the decoding config; 0 = skip
SWEEP_SELECT = "bertscore"   # "bertscore" (primary metric; needs a GPU) | "rouge" ((R1 + RL) / 2)
LEN_MODEL    = dict(a=4.538474, b=0.6144775, lo=232.0, hi=2002.1)       # audit power law (words)
RUN_BERTSCORE, BERTSCORE_MODEL = True, "roberta-large"                   # metric only, never used for training
BASELINES = dict(tfidf=dict(r1=0.5749, r2=0.3186, rL=0.3008, bs=0.8533), lead=dict(r1=0.5222, r2=0.2508, rL=0.2437, bs=0.8321),
                 oracle=dict(r1=0.786, r2=0.6042, rL=0.5341, bs=0.8855))

# ---- Decode-only mode: skip training, load this run's saved adapter weights ----------------------
RESUME_FROM = None           # e.g. Path("/kaggle/input/<dataset>/summ_p6_dec02_lorma.pt") (or best_trainable.pt) -> sweep, VAL, test

# ---- KD teacher export (run #3) -------------------------------------------------------------------
EXPORT_TEACHER    = False    # the student is not a teacher
TEACHER_TOPK      = 8        # top-k logits per gold target token (+ logsumexp), all TR pairs
TEACHER_GEN_PAIRS = 15000    # teacher-generated summaries for this many TR chunks (sequence-level KD)
TEACHER_GEN_MIN   = 90       # time cap (minutes) for the generation part

# ---- Smoke test ----------------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_TRAIN_DOCS, SMOKE_VAL_DOCS = 150, 400, 12

# ---- derived -------------------------------------------------------------------------------------
assert METHOD in HP and SELECT_BY in ("loss", "rouge")
BODY = CHUNK_TOKENS
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} {HP[METHOD]} | chunk {CHUNK_TOKENS} -> target <= {TGT_MAX} | empty keep {EMPTY_KEEP} | "
      f"epochs {EPOCHS} | micro {MICRO_TOKENS} tok, step {STEP_SEQS} seqs | smoke={SMOKE_TEST}")

RUN p6_kd_dec03_student | METHOD=kd {'lr': 0.0003, 'wd': 0.0} | chunk 512 -> target <= 255 | empty keep 0.5 | epochs 4 | micro 12288 tok, step 32 seqs | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- as in Phases 3-4.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_03.ipynb` (Cells 4-11)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# Cell 4 — Model configuration
#
# One config drives three architectures (all registered, all testable):
#   preset_alibi_mla()  -> the NEW architecture (ALiBi positions + Multi-Head Latent Attention)
#   preset_alibi_mha()  -> ablation: ALiBi with ordinary MHA (isolates the ALiBi effect)
#   preset_rope_mha()   -> corrected baseline: RoPE + MHA (model 01 accidentally had NO positions)

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 13
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # only used by mixer_type="attention" (GQA if < n_head)
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU hidden = round64(ff_mult * d_model) = 1408
    d_ff: Optional[int] = None
    max_seq_len: int = 1024           # TRAINING context. ALiBi itself has no length limit.

    # --- components ---
    norm_type: str = "rmsnorm"        # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "alibi"           # "alibi" | "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mixer_type: str = "mla"           # "attention" (MHA/GQA) | "mla"
    kv_latent_dim: int = 128          # d_c: MLA caches only this many numbers per token per layer
    mla_latent_norm: bool = True      # RMSNorm on the latent c_t (as in DeepSeek-V2)
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    channel_mixer_type: str = "mlp"

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("alibi", "rope", "learned", "none")
        assert self.mixer_type in ("attention", "mla")
        if self.mixer_type == "mla":
            # RoPE rotates q and k by a position-dependent matrix that sits BETWEEN W_Q and
            # W_UK, which breaks MLA's weight absorption (DeepSeek needs "decoupled RoPE").
            # ALiBi only adds a bias outside q.k, so it composes cleanly with MLA.
            assert self.pos_type != "rope", "MLA + RoPE needs decoupled RoPE; use ALiBi"
            assert not self.bias, "MLA absorption assumes bias-free up-projections"
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


def preset_alibi_mla(vocab_size=None, max_seq_len=1024):
    # MLA saves ~0.33M attention params per layer vs MHA (d_c=128). 13 layers instead of 12
    # reinvests that saving so the model stays parameter-matched to the 12-layer baseline.
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=13, pos_type="alibi",
                       mixer_type="mla", kv_latent_dim=128, max_seq_len=max_seq_len)

def preset_alibi_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="alibi",
                       mixer_type="attention", max_seq_len=max_seq_len)

def preset_rope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="rope",
                       mixer_type="attention", max_seq_len=max_seq_len)

PRESETS = {"alibi_mla": preset_alibi_mla, "alibi_mha": preset_alibi_mha, "rope_mha": preset_rope_mha}
print(json.dumps(asdict(preset_alibi_mla()), indent=2))

{
  "vocab_size": 32000,
  "n_layer": 13,
  "d_model": 512,
  "n_head": 8,
  "n_kv_head": 8,
  "head_dim": 64,
  "ff_mult": 2.6666666666666665,
  "d_ff": 1408,
  "max_seq_len": 1024,
  "norm_type": "rmsnorm",
  "norm_eps": 1e-05,
  "pos_type": "alibi",
  "rope_theta": 10000.0,
  "mixer_type": "mla",
  "kv_latent_dim": 128,
  "mla_latent_norm": true,
  "mlp_type": "swiglu",
  "channel_mixer_type": "mlp",
  "bias": false,
  "attn_dropout": 0.0,
  "resid_dropout": 0.0,
  "embd_dropout": 0.0,
  "tie_embeddings": true,
  "init_std": 0.02,
  "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers (hand-written)

class LayerNorm(nn.Module):
    """Classic LayerNorm: centre by the mean, scale by the std, learn gain g and bias b."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return (((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype)) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean-centring and no bias; x / rms(x) * g. Statistics computed in fp32."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g


def build_norm(cfg: ModelConfig, d: Optional[int] = None):
    d = d or cfg.d_model
    if cfg.norm_type == "layernorm":
        return LayerNorm(d, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(d, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Positional mechanisms: ALiBi (new) and RoPE (fixed wiring)
#
# ALiBi (Press et al., 2022). No position embeddings at all. Each head h adds a fixed bias
#       s_ij = q_i.k_j / sqrt(d)  -  m_h * (i - j),        j <= i
# Inside the softmax this multiplies attention by exp(-m_h (i-j)): an exponential recency prior.
# Only the distance (i - j) matters: softmax ignores a per-row constant, and
# -m(i-j) = -m*i + m*j, where -m*i is constant along row i.
# Slopes are geometric, m_h = 2^(-8h/H), h = 1..H. For H = 8: 1/2, 1/4, ..., 1/256, so heads
# range from "looks only a few tokens back" to "looks hundreds of tokens back".

def alibi_slopes(n_head: int) -> List[float]:
    def pow2_slopes(n):
        start = 2.0 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    # Non-power-of-2 head counts: the paper's interpolation rule.
    c = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(c) + pow2_slopes(2 * c)[0::2][: n_head - c]


class ALiBi(nn.Module):
    def __init__(self, n_head: int):
        super().__init__()
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head), dtype=torch.float32),
                             persistent=False)

    def bias(self, q_len: int, k_len: int, device) -> torch.Tensor:
        """[1, H, q_len, k_len] float32. Queries are the LAST q_len of the k_len positions
        (so it also works for KV-cached decoding). Includes the causal mask (-inf for j > i)."""
        q_pos = torch.arange(k_len - q_len, k_len, device=device)
        k_pos = torch.arange(k_len, device=device)
        dist = (q_pos[:, None] - k_pos[None, :]).float()          # i - j
        b = -self.slopes.to(device)[:, None, None] * dist          # [H, Tq, Tk]
        b = b.masked_fill(dist < 0, float("-inf"))
        return b.unsqueeze(0)


class RoPE(nn.Module):
    """Rotary embeddings. Table is built for 2x the training length so generation never overflows."""
    def __init__(self, head_dim: int, max_pos: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_pos).float(), inv_freq)   # [P, D/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def tables(self, position_ids: torch.Tensor):
        """position_ids [B, T] -> (cos, sin) each [B, 1, T, D/2]."""
        return self.cos[position_ids].unsqueeze(1), self.sin[position_ids].unsqueeze(1)

    @staticmethod
    def apply(x, cos, sin):
        """x [B, H, T, D]; rotation done in fp32 for stability."""
        dt = x.dtype
        x = x.float()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
        return out.to(dt)

In [8]:
# Cell 7 — Shared attention kernel + standard MHA/GQA token mixer
#
# The Backbone builds ONE additive mask per forward pass (`attn_bias`, shape [B or 1, H or 1, T, S])
# containing ALiBi + causal + padding, and passes it to every layer. Every token mixer then
# calls `attend`. When attn_bias is None we use SDPA's fast is_causal path.

def attend(q, k, v, attn_bias, is_causal, scale, dropout_p=0.0, use_sdpa=True):
    """q [B,H,T,Dq], k [B,H,S,Dq], v [B,H,S,Dv] -> [B,H,T,Dv]."""
    if attn_bias is not None:
        attn_bias = attn_bias.to(q.dtype)          # SDPA requires mask dtype == query dtype
        is_causal = False
    if use_sdpa:
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                              dropout_p=dropout_p, is_causal=is_causal,
                                              scale=scale)
    # Manual reference path (used by the tests to cross-check SDPA).
    scores = (q.float() @ k.float().transpose(-2, -1)) * scale
    if is_causal:
        T, S = q.shape[-2], k.shape[-2]
        scores = scores.masked_fill(
            torch.triu(torch.ones(T, S, dtype=torch.bool, device=q.device), diagonal=S - T + 1),
            float("-inf"))
    if attn_bias is not None:
        scores = scores + attn_bias.float()
    p = F.softmax(scores, dim=-1)
    if dropout_p > 0:
        p = F.dropout(p, dropout_p)
    return (p @ v.float()).to(q.dtype)


class Attention(nn.Module):
    """Multi-head (n_kv_head == n_head) or grouped-query (n_kv_head < n_head) self-attention.
    Cache format: (k, v), each [B, n_kv_head, S, head_dim]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)

    @staticmethod
    def cache_len(cache):
        return cache[0].shape[2]

    def cache_numbers_per_token(self):
        return 2 * self.n_kv_head * self.head_dim

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)
        if rope_cs is not None:                       # FIX: RoPE is now actually applied
            cos, sin = rope_cs
            q, k = RoPE.apply(q, cos, sin), RoPE.apply(k, cos, sin)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        out = attend(q, k, v, attn_bias, is_causal=(attn_bias is None and T > 1),
                     scale=self.scale,
                     dropout_p=self.cfg.attn_dropout if self.training else 0.0,
                     use_sdpa=self.cfg.use_sdpa)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Multi-Head Latent Attention (MLA, DeepSeek-V2), adapted to ALiBi
#
#   c_t      = Norm(W_DKV x_t)          in R^{d_c}        (the ONLY thing we cache)
#   k_t^(h)  = W_UK^(h) c_t,   v_t^(h) = W_UV^(h) c_t    (every head gets its own K, V)
#   q_t^(h)  = W_Q^(h) x_t
#
# KV cache per token per layer: MHA stores 2*H*D_h = 1024 numbers; MLA stores d_c = 128 (8x less).
#
# MLA vs GQA: GQA *ties* heads (groups share identical K,V). MLA gives each head distinct K,V,
# all reconstructed from a shared rank-d_c subspace. With d_c = 2*n_kv*D_h, MLA can exactly
# express GQA (the up-projections just copy slices), so at equal cache size GQA is a special case.
#
# Weight absorption (inference trick, implemented below as `absorb=True`):
#   q_i^(h).k_j^(h) = (W_Q^(h) x_i)^T W_UK^(h) c_j = [W_UK^(h)T q_i^(h)] . c_j
# so we can attend directly against the cached latents without ever building K:
#   q_lat = W_UK^T q,  scores = q_lat . c,  out_lat = softmax(scores) c,  out^(h) = W_UV^(h) out_lat
# RoPE would insert a position-dependent rotation R_{i-j} between W_Q and W_UK and break this;
# ALiBi's bias lives outside the dot product, so absorption is unaffected.
# Training uses the materialised path (simpler, identical maths); a test checks both agree.

class MLAttention(nn.Module):
    """Cache format: c, [B, S, d_c]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        H, Dh, dc = cfg.n_head, cfg.head_dim, cfg.kv_latent_dim
        self.n_head, self.head_dim, self.d_c = H, Dh, dc
        self.scale = 1.0 / math.sqrt(Dh)              # scores are still per-head q.k with dim D_h
        self.q_proj = nn.Linear(cfg.d_model, H * Dh, bias=False)
        self.kv_down = nn.Linear(cfg.d_model, dc, bias=False)          # W_DKV
        self.kv_norm = build_norm(cfg, dc) if cfg.mla_latent_norm else nn.Identity()
        self.k_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UK (all heads)
        self.v_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UV (all heads)
        self.o_proj = nn.Linear(H * Dh, cfg.d_model, bias=False)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)
        self.absorb = False                           # runtime switch (inference only)

    @staticmethod
    def cache_len(cache):
        return cache.shape[1]

    def cache_numbers_per_token(self):
        return self.d_c

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        assert rope_cs is None, "MLA is used with ALiBi (or no positions), not RoPE"
        B, T, _ = x.shape
        H, Dh = self.n_head, self.head_dim
        c = self.kv_norm(self.kv_down(x))                         # [B, T, d_c]
        if kv_cache is not None:
            c = torch.cat([kv_cache, c], dim=1)                   # [B, S, d_c]
        new_cache = c
        S = c.shape[1]
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)      # [B, H, T, Dh]
        is_causal = (attn_bias is None and T > 1)
        dp = self.cfg.attn_dropout if self.training else 0.0

        if not self.absorb:
            k = self.k_up(c).view(B, S, H, Dh).transpose(1, 2)    # [B, H, S, Dh]
            v = self.v_up(c).view(B, S, H, Dh).transpose(1, 2)
            out = attend(q, k, v, attn_bias, is_causal, self.scale, dp, self.cfg.use_sdpa)
        else:
            W_uk = self.k_up.weight.view(H, Dh, self.d_c)         # rows of head h -> W_UK^(h)
            W_uv = self.v_up.weight.view(H, Dh, self.d_c)
            q_lat = torch.einsum("bhtd,hdc->bhtc", q, W_uk.to(q.dtype))   # W_UK^T q
            c_h = c.unsqueeze(1).expand(B, H, S, self.d_c)
            out_lat = attend(q_lat, c_h, c_h, attn_bias, is_causal, self.scale, dp,
                             self.cfg.use_sdpa)                   # [B, H, T, d_c]
            out = torch.einsum("bhtc,hdc->bhtd", out_lat, W_uv.to(out_lat.dtype))

        out = out.transpose(1, 2).contiguous().view(B, T, H * Dh)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [10]:
# Cell 9 — Channel mixers, registries, transformer block

class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))


TOKEN_MIXERS: Dict[str, Any] = {}
CHANNEL_MIXERS: Dict[str, Any] = {}

def register_token_mixer(name):
    def deco(fn):
        TOKEN_MIXERS[name] = fn
        return fn
    return deco

def register_channel_mixer(name):
    def deco(fn):
        CHANNEL_MIXERS[name] = fn
        return fn
    return deco

# FIX: token mixers no longer receive `rope` at build time (model 01 silently dropped it there).
# Positional information now flows through forward(): rope_cs / attn_bias, built in the Backbone.
@register_token_mixer("attention")
def _build_attention(cfg):
    return Attention(cfg)

@register_token_mixer("mla")
def _build_mla(cfg):
    return MLAttention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    return {"swiglu": SwiGLU, "gelu": GELUMLP}[cfg.mlp_type](cfg)


class Block(nn.Module):
    """Pre-norm block: x = x + mixer(norm(x)); x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg)
        self.norm2 = build_norm(cfg)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [11]:
# Cell 10 — Backbone and language model

class Backbone(nn.Module):
    """Embedding -> N blocks -> final norm. Builds positional info and masks ONCE per forward."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.alibi = ALiBi(cfg.n_head) if cfg.pos_type == "alibi" else None
        self.rope = RoPE(cfg.head_dim, 2 * cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(2 * cfg.max_seq_len, cfg.d_model) \
            if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def _attn_bias(self, B, T, past, attention_mask, device):
        """Additive mask [B|1, H|1, T, S] or None (=> SDPA fast causal path)."""
        S = past + T
        need = (self.alibi is not None) or (attention_mask is not None) or (past > 0 and T > 1)
        if not need:
            return None
        if self.alibi is not None:
            bias = self.alibi.bias(T, S, device)                          # ALiBi + causal
        else:
            q_pos = torch.arange(past, S, device=device)
            k_pos = torch.arange(S, device=device)
            bias = torch.zeros(T, S, device=device).masked_fill(
                k_pos[None, :] > q_pos[:, None], float("-inf"))[None, None]
        if attention_mask is not None:
            # Mask padded KEYS. A padded QUERY would then have every key masked -> softmax of
            # all -inf -> NaN, which can leak into later layers through 0*NaN. So let each
            # padded query attend to itself (its output is never used).
            key_pad = ~attention_mask.bool()                                   # [B, S]
            q_idx = torch.arange(past, S, device=device)
            k_idx = torch.arange(S, device=device)
            is_self = q_idx[:, None] == k_idx[None, :]                         # [T, S]
            block = key_pad[:, None, :] & ~is_self[None]                       # [B, T, S]
            pad = torch.zeros(B, 1, T, S, device=device).masked_fill(block[:, None], float("-inf"))
            bias = bias + pad
        return bias

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        B, T = input_ids.shape
        dev = input_ids.device
        past = 0 if kv_caches is None else self.blocks[0].mixer.cache_len(kv_caches[0])
        x = self.tok_emb(input_ids)

        rope_cs = None
        if self.rope is not None or self.pos_emb is not None:
            if attention_mask is not None:       # left padding: real tokens start at position 0
                pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, -T:]
            else:
                pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, T)
            if self.rope is not None:
                rope_cs = self.rope.tables(pos)
            else:
                x = x + self.pos_emb(pos)

        attn_bias = self._attn_bias(B, T, past, attention_mask, dev)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, attn_bias=attn_bias, rope_cs=rope_cs,
                        kv_cache=None if kv_caches is None else kv_caches[i])
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def set_mla_absorb(self, flag: bool):
        for blk in self.backbone.blocks:
            if isinstance(blk.mixer, MLAttention):
                blk.mixer.absorb = flag

    def forward(self, input_ids, targets=None, attention_mask=None, kv_caches=None,
                last_only=False, reduction="mean"):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      kv_caches=kv_caches)
        if last_only:                 # generation: only the last position's logits are needed
            h = h[:, -1:, :]
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=-100, reduction=reduction)
            if reduction == "none":
                loss = loss.view(targets.shape)
        return logits, loss, new_caches

In [12]:
# Cell 11 — Init, parameter counting, optimiser groups, checkpoint I/O

def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, 0.02) everywhere; residual-branch output projections scaled by 1/sqrt(2L)
    so the residual stream's variance does not grow with depth."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            blk.mixer.o_proj.weight.mul_(scale)
            blk.cmix.down_proj.weight.mul_(scale)


def count_params(model: LegalLM) -> Dict[str, int]:
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            total += p.numel()
    emb = model.backbone.tok_emb.weight.numel()
    attn = sum(p.numel() for blk in model.backbone.blocks for p in blk.mixer.parameters())
    return {"total": total, "embedding": emb, "non_embedding": total - emb, "attention": attn}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay only 2-D weight matrices (not embeddings, norm gains or biases)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name and "pos_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, path)


def load_model(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Student and teacher
The student is the Phase-1 Decoder-03 with the blocks in `STUDENT_KEEP_LAYERS`; ALiBi keeps all positional information in the attention bias, so removing blocks does not break positions. The teacher file supplies the top-k logits and the generated summaries, keyed by `"<doc uid>|<chunk>"`.

In [13]:
# Cell 12 — Build the STUDENT from the Phase-1 Decoder-03 checkpoint; load the TEACHER export and check it matches
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lm_cfg = ModelConfig(**_ck["config"])
lm = LegalLM(lm_cfg)
lm.load_state_dict(_ck["model"])
del _ck
lm = lm.to(device).eval()
pc_full = count_params(lm)
print(f"Phase-1 base | pos_type={lm_cfg.pos_type} mixer={lm_cfg.mixer_type} layers={lm_cfg.n_layer} | params {pc_full['total']/1e6:.2f}M")
assert lm_cfg.pos_type == "alibi", "this student recipe is written for the ALiBi model (positions live in the bias, not in the layers)"
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l_full, _ = lm(_x, targets=_y)
assert _l_full.item() < 9.0, "checkpoint looks untrained / mis-loaded"
keep = list(STUDENT_KEEP_LAYERS)
assert keep == sorted(set(keep)) and all(0 <= i < lm_cfg.n_layer for i in keep), "STUDENT_KEEP_LAYERS must be sorted, unique, in range"
lm.backbone.blocks = nn.ModuleList([lm.backbone.blocks[i] for i in keep])
lm_cfg.n_layer = len(keep)
pc = count_params(lm)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l_stu, _ = lm(_x, targets=_y)
print(f"STUDENT: blocks {keep} -> {len(keep)} layers | params {pc['total']/1e6:.2f}M ({100*pc['total']/pc_full['total']:.0f}% of the base) | "
      f"LM loss on a legal sentence {_l_full.item():.3f} (13 blocks) -> {_l_stu.item():.3f} (7 blocks, before fine-tuning)")
assert VOCAB_SIZE < 65536, "token ids are stored as uint16"

TEACHER = torch.load(TEACHER_EXPORT, map_location="cpu", weights_only=False) if TEACHER_EXPORT.exists() else None
if TEACHER is None:
    assert KD_ALPHA == 0 and not SEQ_KD, f"the teacher file {TEACHER_EXPORT} is missing (needed unless KD_ALPHA=0 and SEQ_KD=False)"
    TEACHER = dict(meta={}, topk={}, gen={}); print("no teacher file -> CONTROL run (gold targets only)")
else:
    TM = TEACHER["meta"]
    print(f"TEACHER: {TM['teacher']} ({TM.get('arch', 'decoder')}, {TM['method']}) | {TM['params_total']/1e6:.1f}M params | "
          f"VAL {TM.get('val')} | top-k for {len(TEACHER['topk'])} pairs (k={TM['topk']}) | generations for {len(TEACHER['gen'])} chunks")
    assert TM["params_total"] > pc["total"], f"capstone rule: the teacher must be LARGER than the student ({TM['params_total']/1e6:.1f}M vs {pc['total']/1e6:.1f}M)"
    for k, mine in (("chunk_tokens", CHUNK_TOKENS), ("tgt_max", TGT_MAX), ("min_align_f1", MIN_ALIGN_F1), ("max_full_chars", MAX_FULL_CHARS),
                    ("tag_template", TAG_TEMPLATE), ("val_checksum", VAL_CHECKSUM_EXPECTED)):
        assert TM[k] == mine, f"teacher {k}={TM[k]!r} but this run uses {mine!r}: the targets would not be token-identical"
    print("teacher export matches this run's chunking / targets / split")
if KD_ALPHA == 0 and not SEQ_KD: print("KD_ALPHA = 0 and SEQ_KD = False -> CONTROL run: gold targets only")

Phase-1 base | pos_type=alibi mixer=mla layers=13 | params 53.89M
STUDENT: blocks [0, 2, 4, 6, 8, 10, 12] -> 7 layers | params 36.58M (68% of the base) | LM loss on a legal sentence 1.947 (13 blocks) -> 3.515 (7 blocks, before fine-tuning)
TEACHER: p6_encdec_adapter (encdec, adapter) | 63.5M params | VAL {'r1': 0.5956129781161973, 'r2': 0.3354975130797493, 'rL': 0.3092450440707037, 'bs_p': 0.8561321496963501, 'bs_r': 0.8549532890319824, 'bs_f': 0.8554902672767639} | top-k for 47128 pairs (k=8) | generations for 15000 chunks
teacher export matches this run's chunking / targets / split


## Phase-6 data, the fixed validation split, cleaning
`split_train` and `_h` are copied **verbatim** from the audit (Cell 5); the checksum assert guarantees that every Phase-6 run is scored on the same 200 documents. Cleaning (too-short / too-long summaries, summaries longer than their document) is applied to the **training** part only.

In [14]:
# Cell 13 — Load train / test, split, clean
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def fetch(rel):
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet, or download {P6} and set LOCAL_DATA_DIR.") from ex

ID_KEYS  = ("id", "doc_id", "document_id", "case_id")
DOC_KEYS = ("document", "text", "judgment", "judgement", "source", "case_text", "content")
SUM_KEYS = ("summary", "reference_summary", "abstract", "target", "headnote")
def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default
def _as_text(v): return "\n".join(str(t) for t in v) if isinstance(v, list) else v

def _read_json_any(path):
    txt = Path(path).read_text(encoding="utf-8")
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

def parse_docs(raw, split, has_sum):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): items = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                v = dict(v) if isinstance(v, dict) else {"document": v}
                v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        rid = _first(e, ID_KEYS, i)
        doc = _as_text(_first(e, DOC_KEYS))
        if not isinstance(doc, str): raise ValueError(f"{split} item {rid}: no document field. Keys: {list(e)}")
        summ = _as_text(_first(e, SUM_KEYS)) if has_sum else None
        if has_sum and not isinstance(summ, str): raise ValueError(f"{split} item {rid}: no summary field. Keys: {list(e)}")
        docs.append(dict(uid=f"{split}::{i}", id=rid, doc=doc, summ=summ))
    return docs

def load_split(rel, split, has_sum):
    raw = _read_json_any(fetch(rel))
    print(f"  {rel}: {type(raw).__name__} of {len(raw)}")
    return parse_docs(raw, split, has_sum)

TRAIN = load_split(f"{P6}/Train/train.json", "train", True)
TEST_ALL = load_split(f"{P6}/Test/test.json", "test", False)

# ---- verbatim from the audit (Cell 5) ----
def _norm(t): return re.sub(r"\s+", " ", t).strip().lower()
def _h(t):    return hashlib.md5(_norm(t).encode("utf-8")).hexdigest()
def split_train(train, val_n=VAL_N, salt=VAL_SALT):
    # Whole duplicate-document groups go to one side; group order = md5(salt :: normalised-text hash).
    groups = defaultdict(list)
    for d in train: groups[_h(d["doc"])].append(d)
    order = sorted(groups, key=lambda g: hashlib.md5(f"{salt}::{g}".encode()).hexdigest())
    val, val_keys = [], set()
    for g in order:
        if len(val) >= val_n: break
        val.extend(groups[g]); val_keys.add(g)
    tr = [d for d in train if _h(d["doc"]) not in val_keys]
    checksum = hashlib.md5("|".join(sorted(str(d["id"]) for d in val)).encode()).hexdigest()[:12]
    return tr, val, checksum
# ------------------------------------------

TR_ALL, VAL_ALL, _cs = split_train(TRAIN)
print(f"split: TR {len(TR_ALL)} | VAL {len(VAL_ALL)} | checksum {_cs}")
assert _cs == VAL_CHECKSUM_EXPECTED, f"VAL split differs from the audit ({_cs} != {VAL_CHECKSUM_EXPECTED})"

def _clean(docs):
    keep, drop = [], Counter()
    seen = set()
    for d in docs:
        sw, dw = len(d["summ"].split()), len(d["doc"].split())
        key = (_h(d["doc"]), _h(d["summ"]))
        if sw < MIN_SUMM_WORDS: drop["short summary"] += 1; continue
        if sw > MAX_SUMM_WORDS: drop["very long summary"] += 1; continue
        if sw > dw: drop["summary longer than document"] += 1; continue
        if key in seen: drop["exact duplicate pair"] += 1; continue
        seen.add(key); keep.append(d)
    return keep, dict(drop)
TR_ALL, _dropped = _clean(TR_ALL)
print(f"training docs after cleaning: {len(TR_ALL)} | dropped {_dropped}")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

  Phase_6_Summ/Train/train.json: list of 7030


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

  Phase_6_Summ/Test/test.json: list of 100
split: TR 6830 | VAL 200 | checksum 5720e140c147
training docs after cleaning: 6759 | dropped {'exact duplicate pair': 25, 'very long summary': 43, 'summary longer than document': 1, 'short summary': 2}


## Sentence alignment → chunk-level training pairs
`split_sentences`, `content` and `align_doc` are the audit's (Cell 8), unchanged. For every training document: tokenise it **with character offsets**, cut it into 512-token chunks, align each summary sentence to its best document sentence, and give the sentence to the chunk containing that source sentence. A chunk's target is its sentences **in summary order**, cut at a sentence boundary to `TGT_MAX` tokens (a single over-long sentence is cut and gets no `[EOS]`, so the model never learns to stop mid-sentence). Tokenisation and alignment run in `N_PROC` processes and are cached.

In [15]:
# Cell 14 — Sentence splitter + aligner (verbatim from the audit, Cell 8)
_WRE = re.compile(r"[a-z0-9]+")
def words(t): return _WRE.findall(t.lower())
_ABBR = (r"mr|mrs|ms|dr|hon'ble|honble|no|nos|sr|jr|ltd|pvt|co|corp|inc|vs|v|viz|i\.e|e\.g|etc|cf|art|arts|sec|secs|s|ss|cl|"
         r"para|paras|p|pp|vol|ors|anr|govt|st|rs|prof|smt|shri|sri|ch|app|crl|cr|civ|j|jj|cj|mad|bom|cal|del|all|scc|scr|air|"
         r"u/s|r/w|w\.e\.f|addl|asstt|dist|distt|dept|reg|regd|ft|sq|nos|approx|ibid|supra|id|cri|lj|cri\.l\.j|misc|wp|slp")
_ABBR_END = re.compile(r"(?:\b(?:" + _ABBR + r")|\b[A-Za-z])\.$|^\s*\(?\d{1,3}[.)]$", re.I)
_SPLIT = re.compile(r"(?<=[.?!])[\"')\]]*\s+(?=[\"'(\[]?[A-Z0-9])|\n\s*\n|\n(?=\s*(?:\d{1,3}[.)]\s|\([a-z0-9]{1,4}\)\s))")

def split_sentences(text):
    # Returns [(char_start, sentence)]. Pieces ending in an abbreviation / initial / short number are merged forward.
    out, start = [], 0
    pieces = []
    for m in _SPLIT.finditer(text):
        pieces.append((start, text[start:m.start()])); start = m.end()
    pieces.append((start, text[start:]))
    buf_s, buf = None, ""
    for s, p in pieces:
        if buf_s is None: buf_s, buf = s, p
        else: buf = buf + " " + p
        if not _ABBR_END.search(buf.rstrip()):
            if buf.strip(): out.append((buf_s, buf.strip()))
            buf_s, buf = None, ""
    if buf_s is not None and buf.strip(): out.append((buf_s, buf.strip()))
    return out

_STOP = set(("the a an of to in and or for on by with at from as is was were be been are that this these those it its which who "
             "whom whose has have had not no but if than then so such any all also there their they he she his her him we our us you "
             "your i me my shall may can could would should will said upon into under over between after before against within "
             "without about being do does did same other further thereof therein herein hereby whereas").split())
def content(t): return {w for w in _WRE.findall(t.lower()) if w not in _STOP and len(w) > 1}

def align_doc(doc, summ):
    ds, ss = split_sentences(doc), split_sentences(summ)
    dsets = [content(t) for _, t in ds]
    inv = defaultdict(list)
    for j, st in enumerate(dsets):
        for w in st: inv[w].append(j)
    res = []
    for k, (_, t) in enumerate(ss):
        q = content(t)
        if len(q) < 2: continue
        hits = Counter()
        for w in q:
            for j in inv.get(w, ()): hits[j] += 1
        best_j, best_f = -1, 0.0
        for j, c in hits.items():
            f = 2 * c / (len(q) + len(dsets[j]))
            if f > best_f: best_j, best_f = j, f
        if best_j >= 0:
            res.append(dict(k=k, j=best_j, f=best_f, pos=ds[best_j][0] / max(len(doc), 1), char=ds[best_j][0], words=len(t.split())))
    return ds, ss, res

In [16]:
# Cell 15 — Tokenise documents into chunks (+ chunk targets for documents with a summary); cache; prompt pieces
def cut_text(t):
    if len(t) <= MAX_FULL_CHARS: return t
    h = 2 * MAX_FULL_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_FULL_CHARS - h):]

def _fit_target(sents):
    # sentences (summary order) -> (token ids <= TGT_MAX, truncated?) ; whole sentences while they fit
    chosen, ids = [], []
    for s in sents:
        cand = tokenizer.encode(" ".join(chosen + [s]))
        if len(cand) <= TGT_MAX: chosen.append(s); ids = cand
        else: break
    if chosen: return ids, False
    return tokenizer.encode(sents[0])[:TGT_MAX], True

_DOCS_W, _WITH_TGT = None, False
def _work(i):
    d = _DOCS_W[i]
    text = cut_text(d["doc"])
    ids, offs = tokenizer.encode(text, return_offsets=True)
    ids = list(ids)
    if not ids: ids, offs = [EOS_ID], [(0, 0)]
    starts = list(range(0, len(ids), BODY))
    cchar = [offs[s][0] for s in starts]
    chunks = [np.asarray(ids[s:s + BODY], dtype=np.uint16) for s in starts]
    tgts, trunc = None, None
    if _WITH_TGT and d["summ"] is not None:
        _, ss, res = align_doc(text, d["summ"])
        per = defaultdict(set)
        for r in res:
            if r["f"] >= MIN_ALIGN_F1:
                per[max(0, bisect.bisect_right(cchar, r["char"]) - 1)].add(r["k"])
        tgts, trunc = [], []
        for j in range(len(starts)):
            if not per.get(j): tgts.append(None); trunc.append(False); continue
            t_ids, tr = _fit_target([ss[k][1] for k in sorted(per[j])])
            tgts.append(np.asarray(t_ids, dtype=np.uint16)); trunc.append(tr)
    return dict(chunks=chunks, tgts=tgts, trunc=trunc, ntok=len(ids), words=len(d["doc"].split()))

def prepare_docs(docs, cache_name, with_targets):
    global _DOCS_W, _WITH_TGT
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"p6_{cache_name}_v{VOCAB_SIZE}_c{CHUNK_TOKENS}_t{TGT_MAX}_a{MIN_ALIGN_F1}_m{MAX_FULL_CHARS}_{int(with_targets)}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    key = lambda d: (d["uid"], len(d["doc"]), len(d["summ"] or ""))
    todo = [i for i, d in enumerate(docs) if cache.get(d["uid"], (None,))[0] != key(d)]
    if todo:
        _DOCS_W, _WITH_TGT = docs, with_targets
        order = sorted(todo, key=lambda i: -len(docs[i]["doc"]))
        res = None
        if N_PROC > 1:
            try:
                with mp.get_context("fork").Pool(N_PROC) as pool:
                    res = list(tqdm(pool.imap(_work, order, chunksize=8), total=len(order), desc=f"prepare {cache_name}", leave=False))
            except Exception as ex:
                print("parallel preparation failed (", type(ex).__name__, ex, ") -> serial"); res = None
        if res is None:
            res = [_work(i) for i in tqdm(order, desc=f"prepare {cache_name}", leave=False)]
        for i, r in zip(order, res): cache[docs[i]["uid"]] = (key(docs[i]), r)
        pickle.dump(cache, open(cp, "wb"))
    for d in docs: d.update(cache[d["uid"]][1])
    return docs

SEP_IDS = tokenizer.encode(SEP_TEXT)
_TAG_CACHE = {}
def tag_ids(j, n):
    k = (j, n)
    if k not in _TAG_CACHE: _TAG_CACHE[k] = tokenizer.encode(TAG_TEMPLATE.format(i=j + 1, n=n))
    return _TAG_CACHE[k]
def prompt_ids(d, j):
    return [BOS_ID] + tag_ids(j, len(d["chunks"])) + d["chunks"][j].astype(np.int64).tolist() + SEP_IDS
MAX_PROMPT = 1 + len(tag_ids(998, 999)) + BODY + len(SEP_IDS)
assert MAX_PROMPT + TGT_MAX + 1 <= lm_cfg.max_seq_len, f"prompt {MAX_PROMPT} + target {TGT_MAX} exceeds the RoPE table {lm_cfg.max_seq_len}"
print(f"prompt <= {MAX_PROMPT} tokens + target <= {TGT_MAX} + EOS -> sequences <= {MAX_PROMPT + TGT_MAX + 1} (context {lm_cfg.max_seq_len})")

def pair_stats(docs, title):
    n_ch = np.asarray([len(d["chunks"]) for d in docs])
    t = [len(x) for d in docs for x in d["tgts"] if x is not None]
    emp = np.mean([x is None for d in docs for x in d["tgts"]])
    tr = np.mean([f for d in docs for x, f in zip(d["tgts"], d["trunc"]) if x is not None])
    print(f"{title}: {len(docs)} docs | chunks/doc mean {n_ch.mean():.1f} median {np.median(n_ch):.0f} | chunk pairs with a target {len(t)} "
          f"| empty chunks {100 * emp:.1f}% | target tokens median {np.median(t):.0f} p90 {np.percentile(t, 90):.0f} | "
          f"single-sentence cut {100 * tr:.1f}%")

prompt <= 529 tokens + target <= 255 + EOS -> sequences <= 785 (context 1024)


## Student model, KD batches and loss, generation
* Training pairs are `(doc, chunk, kind)`: `kind 0` = gold target (+ the teacher's top-k logits when they exist), `kind 1` = the teacher's generated summary (sequence-level KD, CE only).
* The loss is computed only at target positions. For gold tokens with teacher logits: `(1 − α)·CE + α·KL_{k+1}` (the partition KL above); otherwise CE. Cell 19 checks that the KL is ≥ 0 and exactly 0 when the student *is* the teacher.
* Generation: the Phase-1 Dec-03 backbone with left-padded prompts and the (MLA latent) KV cache, same decoding controls as runs #1–#3.

In [17]:
# Cell 16 — The student model (full fine-tuning of the truncated Dec-03)
def prefix_kv(model, B): return None                    # the generation code below is shared with run #3 (no prefix here)

def build_model():
    model = copy.deepcopy(lm).to(device)
    for n, p in model.named_parameters():
        p.requires_grad = not (FREEZE_EMBEDDINGS and "tok_emb" in n)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"student params: total {tot/1e6:.2f}M | trainable {tr/1e6:.2f}M ({100*tr/tot:.0f}%)")
    model.n_total, model.n_trainable = tot, tr
    return model

def backbone_hidden(model, ids, mask=None, pos=None):
    h, _ = model.backbone(ids)                          # right padding + causal ALiBi bias -> no padding mask needed
    return h

In [18]:
# Cell 17 — Training pairs (gold + teacher generations), micro-batches, KD loss
def _tkey(d, j): return f"{d['uid']}|{j}"

def make_pairs(docs, seed, empty_keep=EMPTY_KEEP):
    # -> list of (doc index, chunk index, kind). kind 0: gold target (or EOS-only empty chunk); kind 1: teacher generation.
    rng = random.Random(seed); pairs = []
    for di, d in enumerate(docs):
        for j, t in enumerate(d["tgts"]):
            if t is not None or rng.random() < empty_keep: pairs.append((di, j, 0))
            if SEQ_KD and _tkey(d, j) in TEACHER["gen"]: pairs.append((di, j, 1))
    return pairs

def target_of(docs, pair):
    di, j, kind = pair; d = docs[di]
    if kind == 1:
        g = TEACHER["gen"][_tkey(d, j)].astype(np.int64).tolist()
        return g + ([EOS_ID] if len(g) < TGT_MAX else [])
    t = d["tgts"][j]
    return ([] if t is None else t.astype(np.int64).tolist()) + ([] if (t is not None and d["trunc"][j]) else [EOS_ID])

def make_microbatches(docs, pairs, shuffle, seed=0, budget=None):
    budget = budget or MICRO_TOKENS
    lens = [len(prompt_ids(docs[p[0]], p[1])) + len(target_of(docs, p)) for p in pairs]
    idx = list(range(len(pairs))); rng = random.Random(seed)
    if shuffle: rng.shuffle(idx)
    batches, pool = [], 2000
    for p in range(0, len(idx), pool):
        part = sorted(idx[p:p + pool], key=lambda i: lens[i])
        cur, longest = [], 0
        for i in part:
            if cur and (len(cur) + 1) * max(longest, lens[i]) > budget:
                batches.append(cur); cur, longest = [], 0
            cur.append(i); longest = max(longest, lens[i])
        if cur: batches.append(cur)
    if shuffle: rng.shuffle(batches)
    return [[pairs[i] for i in b] for b in batches]

KD_STATS = Counter()
def build_batch(docs, pairs, with_teacher=True):
    seqs, labs, tv, ti, tl, has = [], [], [], [], [], []
    K = TEACHER["meta"].get("topk", 1)
    for pr in pairs:
        p, t = prompt_ids(docs[pr[0]], pr[1]), target_of(docs, pr)
        full = p + t
        seqs.append(full[:-1]); labs.append([-100] * (len(p) - 1) + t)
        e = TEACHER["topk"].get(_tkey(docs[pr[0]], pr[1])) if (with_teacher and KD_ALPHA > 0 and pr[2] == 0) else None
        if e is not None and len(e[2]) != len(t): KD_STATS["length_mismatch"] += 1; e = None
        if e is not None:
            tv.append(e[0].astype(np.float32)); ti.append(e[1].astype(np.int64)); tl.append(e[2]); has.append(np.ones(len(t), bool))
            KD_STATS["with_logits"] += 1
        else:
            tv.append(np.zeros((len(t), K), np.float32)); ti.append(np.zeros((len(t), K), np.int64))
            tl.append(np.zeros(len(t), np.float32)); has.append(np.zeros(len(t), bool))
    T = max(len(s) for s in seqs)
    x = np.full((len(seqs), T), PAD_ID, dtype=np.int64); y = np.full((len(seqs), T), -100, dtype=np.int64)
    for i, (s, l) in enumerate(zip(seqs, labs)):
        x[i, :len(s)] = s; y[i, :len(l)] = l
    g = lambda a: torch.from_numpy(a).to(device)
    return dict(x=g(x), y=g(y), tv=g(np.concatenate(tv)), ti=g(np.concatenate(ti)), tl=g(np.concatenate(tl)),
                has=g(np.concatenate(has)), ntok=int(x.size))

def kd_kl(logits, tv, ti, tl):
    # exact KL between teacher and student on the partition {top-k tokens, rest}; logits fp32 [N, V]
    logq = F.log_softmax(logits, -1)
    lq = logq.gather(1, ti)                                       # log q_k
    lp = tv - tl[:, None]                                         # log p_k (true teacher probabilities)
    p = lp.exp()
    r = (1 - p.sum(1)).clamp(min=1e-6)
    q_rest = (1 - lq.exp().sum(1)).clamp(min=1e-6)
    return (p * (lp - lq)).sum(1) + r * (r.log() - q_rest.log())

def batch_loss(model, bt):
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        h = backbone_hidden(model, bt["x"])
        sel = bt["y"] != -100                                     # row-major order == the order of the concatenated teacher arrays
        logits = model.lm_head(h[sel])
    logits = logits.float(); y = bt["y"][sel]
    loss = F.cross_entropy(logits, y, reduction="none")
    if KD_ALPHA > 0 and bool(bt["has"].any()):
        m = bt["has"]
        kl = kd_kl(logits[m], bt["tv"][m], bt["ti"][m], bt["tl"][m])
        loss = loss.clone(); loss[m] = (1 - KD_ALPHA) * loss[m] + KD_ALPHA * kl
    return loss.sum(), int(sel.sum().item())

@torch.no_grad()
def eval_loss(model, docs, pairs):
    # gold-target CE only (comparable with runs #1-#3)
    model.eval(); tot, n = 0.0, 0
    for b in make_microbatches(docs, [p for p in pairs if p[2] == 0], shuffle=False, budget=2 * MICRO_TOKENS):
        bt = build_batch(docs, b, with_teacher=False)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            h = backbone_hidden(model, bt["x"]); sel = bt["y"] != -100; lg = model.lm_head(h[sel])
        tot += F.cross_entropy(lg.float(), bt["y"][sel], reduction="sum").item(); n += int(sel.sum().item())
    return tot / max(n, 1)

In [19]:
# Cell 18 — Batched greedy generation with KV cache, document summarisation, ROUGE
@torch.no_grad()
def generate(model, prompts, max_new, min_new=None, no_repeat_ngram=0, rep_penalty=1.0, amp=True, eos_threshold=0.0):
    # prompts: list of token-id lists (left-padded here). The prefix (if any) is the initial KV cache. Returns ids without EOS.
    model.eval(); B = len(prompts); L = max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=device); mask = torch.zeros((B, L), dtype=torch.long, device=device)
    for i, p in enumerate(prompts):
        ids[i, L - len(p):] = torch.tensor(p, device=device); mask[i, L - len(p):] = 1
    min_new_t = torch.tensor(min_new if min_new is not None else [0] * B, device=device)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP and amp):
        pk = prefix_kv(model, B)
        if pk is not None: mask = torch.cat([torch.ones((B, pk[0][0].shape[2]), dtype=mask.dtype, device=device), mask], 1)
        h, caches = model.backbone(ids, attention_mask=mask, kv_caches=pk)
        logits = model.lm_head(h[:, -1]).float()
    out = [[] for _ in range(B)]; done = [False] * B; bans = [defaultdict(set) for _ in range(B)]
    seen = torch.zeros((B, logits.shape[-1]), dtype=torch.bool, device=device); n = no_repeat_ngram
    for t in range(max_new):
        lg = logits
        if rep_penalty != 1.0:
            lg = torch.where(seen, torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty), lg)
        lg[:, PAD_ID] = float("-inf"); lg[:, BOS_ID] = float("-inf")
        if eos_threshold > 0:
            lg[:, EOS_ID] = torch.where(torch.softmax(lg, -1)[:, EOS_ID] < eos_threshold, torch.full_like(lg[:, EOS_ID], float("-inf")), lg[:, EOS_ID])
        lg[:, EOS_ID] = torch.where(t < min_new_t, torch.full_like(lg[:, EOS_ID], float("-inf")), lg[:, EOS_ID])
        if n > 0 and t >= n - 1:
            rows, cols = [], []
            for b in range(B):
                if done[b]: continue
                for tok in bans[b].get(tuple(out[b][-(n - 1):]) if n > 1 else (), ()): rows.append(b); cols.append(tok)
            if rows: lg[rows, cols] = float("-inf")
        nl = lg.argmax(-1).tolist()
        for b in range(B):
            if done[b]: nl[b] = PAD_ID; continue
            if nl[b] == EOS_ID: done[b] = True; continue
            out[b].append(nl[b])
            if n > 0 and len(out[b]) >= n: bans[b][tuple(out[b][-n:-1]) if n > 1 else ()].add(out[b][-1])
        if all(done): break
        nxt = torch.tensor(nl, device=device)
        seen[torch.arange(B, device=device), nxt] = True
        mask = torch.cat([mask, torch.ones((B, 1), dtype=mask.dtype, device=device)], 1)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP and amp):
            h, caches = model.backbone(nxt[:, None], attention_mask=mask, kv_caches=caches)
            logits = model.lm_head(h[:, -1]).float()
    return out

def select_chunks(n, k):
    # Head + tail + evenly spread middle (the audit's U-shaped source positions).
    if n <= k: return list(range(n))
    h = max(1, k // 4); t = max(1, k // 4); m = k - h - t
    mid = np.linspace(h, n - t - 1, m).round().astype(int).tolist() if m > 0 else []
    return sorted(set(list(range(h)) + mid + list(range(n - t, n))))

def predict_len_words(doc_words):
    return float(np.clip(LEN_MODEL["a"] * max(doc_words, 1) ** LEN_MODEL["b"], LEN_MODEL["lo"], LEN_MODEL["hi"]))

def _dedup(text):
    seen, out = set(), []
    for _, s in split_sentences(text):
        k = " ".join(words(s))
        if k and k in seen: continue
        seen.add(k); out.append(s)
    return " ".join(out)

def _dedup_parts(segs):
    # sentence-level de-duplication across the chunk outputs (keeps chunk boundaries)
    seen, out = set(), []
    for t in segs:
        keep = []
        for _, s in split_sentences(t):
            k = " ".join(words(s))
            if k and k in seen: continue
            seen.add(k); keep.append(s)
        if keep: out.append(" ".join(keep))
    return out

def _cut_words(text, n):
    # first n words, ending at the last sentence end in the kept part if that keeps >= 60 % of n
    w = text.split()
    if len(w) <= n: return text
    head = " ".join(w[:n]); m = max(head.rfind(". "), head.rfind("? "), head.rfind("! "))
    if head.endswith((".", "?", "!")): return head
    return head[:m + 1] if m >= 0 and len(head[:m + 1].split()) >= 0.6 * n else head

def _trim(segs, budget, mode):
    tot = sum(len(t.split()) for t in segs)
    if tot <= budget: return " ".join(segs)
    if mode == "tail": return " ".join(" ".join(segs).split()[:budget])
    out = [_cut_words(t, max(1, int(len(t.split()) * budget / tot))) for t in segs]     # every chunk keeps the same share
    return " ".join(x for x in out if x)

LAST_GEN_STATS = {}
def summarize_docs(model, docs, gen=None, return_parts=False):
    g = dict(GEN, **(gen or {}))
    jobs = []                                                               # (doc index, chunk index)
    for di, d in enumerate(docs):
        for j in select_chunks(len(d["chunks"]), g["max_chunks"]): jobs.append((di, j))
    prompts = [prompt_ids(docs[di], j) for di, j in jobs]
    first = {di: min(j for dj, j in jobs if dj == di) for di in range(len(docs))}
    order = sorted(range(len(jobs)), key=lambda i: len(prompts[i]))
    outs = [None] * len(jobs); t0 = time.time()
    for s in range(0, len(order), g["batch"]):
        bi = order[s:s + g["batch"]]
        mins = [max(g["min_new_all"], g["min_new_first"] if jobs[i][1] == first[jobs[i][0]] else 0) for i in bi]
        res = generate(model, [prompts[i] for i in bi], g["max_new"], mins, g["no_repeat_ngram"], g["rep_penalty"], eos_threshold=g["eos_threshold"])
        for i, r in zip(bi, res): outs[i] = r
    parts = defaultdict(list)
    for (di, j), o in zip(jobs, outs): parts[di].append((j, tokenizer.decode(o).strip()))
    LAST_GEN_STATS.update(chunks=len(jobs), empty_share=float(np.mean([len(o) == 0 for o in outs])),
                          tokens_per_chunk=float(np.mean([len(o) for o in outs])), hit_max_share=float(np.mean([len(o) >= g["max_new"] for o in outs])))
    summaries = []
    for di, d in enumerate(docs):
        segs = [t for _, t in sorted(parts[di]) if t]
        if g["dedup"]: segs = _dedup_parts(segs)
        budget = int(round(g["len_factor"] * predict_len_words(d["words"])))
        summaries.append(_trim(segs, budget, g["trim"]))
    secs = time.time() - t0
    return (summaries, secs, parts) if return_parts else (summaries, secs)

# ---- ROUGE: fast exact implementation verified against rouge_score (as in the audit, Cell 9) ----
try:
    from rouge_score import rouge_scorer
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score"], check=True)
    from rouge_score import rouge_scorer
_RS = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
def rtok(text): return _RS._tokenizer.tokenize(text)
def lcs_len(a, b):
    if not a or not b: return 0
    masks = {}
    for i, x in enumerate(a): masks[x] = masks.get(x, 0) | (1 << i)
    full = (1 << len(a)) - 1; V = full
    for y in b:
        U = V & masks.get(y, 0)
        V = ((V + U) | (V - U)) & full
    return len(a) - bin(V).count("1")
def _f(o, nc, nr):
    p, r = (o / nc if nc else 0.0), (o / nr if nr else 0.0)
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)
def rouge(cand, ref):
    c, r = rtok(cand), rtok(ref); out = {}
    for n in (1, 2):
        cc, rc = Counter(zip(*[c[i:] for i in range(n)])), Counter(zip(*[r[i:] for i in range(n)]))
        out[f"r{n}"] = _f(sum(min(v, rc[k]) for k, v in cc.items()), sum(cc.values()), sum(rc.values()))
    out["rL"] = _f(lcs_len(r, c), len(c), len(r))
    return out
_c, _r = "the appellant was convicted under section 302 by the court", "the appellant was convicted by the sessions court under section 302"
assert all(abs(_RS.score(_r, _c)[k].fmeasure - rouge(_c, _r)[k2][2]) < 1e-9 for k, k2 in (("rouge1", "r1"), ("rouge2", "r2"), ("rougeL", "rL")))

def rouge_eval(cands, refs):
    R = defaultdict(list)
    for c, r in zip(cands, refs):
        for k, v in rouge(c, r).items(): R[k].append(v)
    return {k: dict(p=float(np.mean([x[0] for x in v])), r=float(np.mean([x[1] for x in v])), f=float(np.mean([x[2] for x in v])))
            for k, v in R.items()}, {k: np.asarray([x[2] for x in v]) for k, v in R.items()}

In [20]:
# Cell 19 — Self-tests: (a) the partition KL; (b) batched + cached + left-padded greedy == reference greedy (student)
_z = torch.randn(6, VOCAB_SIZE) * 3
_v, _i = _z.topk(8, -1); _lse = torch.logsumexp(_z, -1)
assert kd_kl(_z, _v, _i, _lse).abs().max().item() < 1e-4, "KL must be 0 when the student equals the teacher"
assert (kd_kl(torch.randn(6, VOCAB_SIZE), _v, _i, _lse) >= -1e-5).all(), "KL must be >= 0"
print("partition-KL self-test OK (0 for identical distributions, >= 0 otherwise)")

def _greedy_ref(model, p, steps):
    seq = list(p); out = []
    for _ in range(steps):
        with torch.no_grad():
            h, _ = model.backbone(torch.tensor([seq], device=device))
            lg = model.lm_head(h[:, -1]).float()[0]
        lg[PAD_ID] = float("-inf"); lg[BOS_ID] = float("-inf")
        t = int(lg.argmax())
        if t == EOS_ID: break
        out.append(t); seq.append(t)
    return out
_tf32 = torch.backends.cuda.matmul.allow_tf32
torch.backends.cuda.matmul.allow_tf32 = False
_tp = [[BOS_ID] + tokenizer.encode("The appellant was convicted by the Sessions Court and"), [BOS_ID] + tokenizer.encode("Held:"),
       [BOS_ID] + tokenizer.encode(VAL_ALL[0]["doc"][:2400])[:420]]
_b = generate(lm, _tp, 24, amp=False)
for p, g in zip(_tp, _b):
    r = _greedy_ref(lm, p, 24)
    assert g == r, f"cached batched generation differs from the reference:\n{g}\n{r}"
torch.backends.cuda.matmul.allow_tf32 = _tf32
print(f"generation self-test OK: batched + MLA cache + left padding == reference greedy (prompts {[len(p) for p in _tp]})")

partition-KL self-test OK (0 for identical distributions, >= 0 otherwise)
generation self-test OK: batched + MLA cache + left padding == reference greedy (prompts [10, 4, 421])


## Build the data and train

In [21]:
# Cell 20 — Prepare data, build the model, train
seed_everything(SEED)
if SMOKE_TEST:
    _r = random.Random(SEED)
    train_docs = _r.sample(TR_ALL, min(SMOKE_TRAIN_DOCS, len(TR_ALL))); val_docs = VAL_ALL[:SMOKE_VAL_DOCS]
else:
    train_docs, val_docs = TR_ALL, VAL_ALL
_t0 = time.time()
if RESUME_FROM is None or EXPORT_TEACHER:                                   # decode-only runs need the TR chunks only for the export
    prepare_docs(train_docs, "tr_smoke" if SMOKE_TEST else "tr", True); pair_stats(train_docs, "TR")
prepare_docs(val_docs, "val_smoke" if SMOKE_TEST else "val", True)
print(f"preparation {time.time() - _t0:.0f}s")
pair_stats(val_docs, "VAL")
val_pairs = make_pairs(val_docs, seed=0, empty_keep=EMPTY_KEEP)

def param_groups(model):
    hp = HP[METHOD]
    ps = [p for p in model.parameters() if p.requires_grad]
    return [dict(params=ps, lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"])]

def quick_rouge(model, docs, n):
    if not n: return None
    s, _ = summarize_docs(model, docs[:n])
    m, _ = rouge_eval(s, [d["summ"] for d in docs[:n]])
    return m

def train_model(model, epochs, max_steps=None):
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    n_pairs = len(make_pairs(train_docs, seed=SEED + 1))
    steps_per_epoch = math.ceil(n_pairs / STEP_SEQS); total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    print(f"~{n_pairs} training sequences/epoch | ~{steps_per_epoch} steps/epoch ({STEP_SEQS} seqs/step) | total {total} steps")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=None, epoch=0, state=None); t_train = time.time(); tok_seen = 0
    v0 = eval_loss(model, val_docs, val_pairs); print(f"VAL loss before training (Phase-1 model, target tokens): {v0:.4f}")
    for ep in range(1, epochs + 1):
        pairs = make_pairs(train_docs, seed=SEED + ep)
        batches = make_microbatches(train_docs, pairs, shuffle=True, seed=SEED + ep)
        model.train(); t0 = time.time(); ep_sum, ep_tok, acc, acc_tok, acc_loss = 0.0, 0, 0, 0, None
        for bi, b in enumerate(batches):
            bt = build_batch(train_docs, b)
            l, k = batch_loss(model, bt)
            # normalise by the target tokens of the whole step: sum-losses are divided by an estimate (k * STEP_SEQS / len(b))
            scaler.scale(l / max(1.0, k * STEP_SEQS / len(b))).backward()
            ep_sum += l.item(); ep_tok += k; tok_seen += bt["ntok"]; acc += len(b)
            losses.append(l.item() / max(k, 1))
            if acc >= STEP_SEQS or bi == len(batches) - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1; acc = 0
                if step == 20:
                    per = (time.time() - t0) / 20
                    print(f"  >> speed: {per:.2f} s/step | projected {per * steps_per_epoch / 60:.1f} min/epoch "
                          f"({per * steps_per_epoch * epochs / 60:.0f} min for {epochs} epochs, plus evaluation)"
                          + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
                if step % 100 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-300:]):.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
                if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                    print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping"); done = True; break
        vl = eval_loss(model, val_docs, val_pairs)
        qr = quick_rouge(model, val_docs, min(VAL_GEN_DOCS_EPOCH, len(val_docs)))
        rec = dict(epoch=ep, train_loss=ep_sum / max(ep_tok, 1), val_loss=vl, seconds=time.time() - t0, steps=step)
        if qr: rec.update(q_r1=qr["r1"]["f"], q_r2=qr["r2"]["f"], q_rL=qr["rL"]["f"])
        score = -vl if SELECT_BY == "loss" else (qr["r1"]["f"] + qr["rL"]["f"]) / 2
        if best["score"] is None or score > best["score"]:
            best = dict(score=score, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
            torch.save(best, RUN_DIR / "best_trainable.pt")
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step, tokens_seen=tok_seen, val_loss_init=v0,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    return history, best, stats

_np = make_pairs(train_docs, seed=SEED)
print(f"training pairs: {sum(p[2] == 0 for p in _np)} gold + {sum(p[2] == 1 for p in _np)} teacher generations (SEQ_KD={SEQ_KD}) | "
      f"gold pairs with teacher logits: {sum(1 for p in _np if p[2] == 0 and _tkey(train_docs[p[0]], p[1]) in TEACHER['topk'])}")
model = build_model()
if RESUME_FROM is None:
    history, best, stats = train_model(model, EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
    model.load_state_dict(best["state"], strict=False)
    print(f"\nbest epoch {best['epoch']} (by {SELECT_BY}) | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB | KD batches: {dict(KD_STATS)}")
else:
    _r = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)
    _st = _r.get("trainable") or _r.get("state")
    _miss = [n for n, p in model.named_parameters() if p.requires_grad and n not in _st]
    assert not _miss, f"{RESUME_FROM} lacks trainable weights: {_miss[:3]}"
    model.load_state_dict(_st, strict=False)
    _log = _r.get("log", {})
    history, best = _log.get("history", []), dict(epoch=_log.get("best_epoch", 0), state=_st)
    stats = dict(_log.get("stats", {}), resumed_from=str(RESUME_FROM))
    print(f"DECODE-ONLY: loaded {len(_st)} trainable tensors from {RESUME_FROM} (best epoch {best['epoch']})")
    if val_docs: print(f"VAL loss of the loaded weights: {eval_loss(model, val_docs, val_pairs):.4f}")

prepare tr:   0%|          | 0/6759 [00:00<?, ?it/s]

TR: 6759 docs | chunks/doc mean 10.2 median 8 | chunk pairs with a target 47132 | empty chunks 31.8% | target tokens median 115 p90 231 | single-sentence cut 0.4%


prepare val:   0%|          | 0/200 [00:00<?, ?it/s]

preparation 126s
VAL: 200 docs | chunks/doc mean 11.0 median 9 | chunk pairs with a target 1525 | empty chunks 30.6% | target tokens median 109 p90 232 | single-sentence cut 0.5%
training pairs: 58165 gold + 15000 teacher generations (SEQ_KD=True) | gold pairs with teacher logits: 47102
student params: total 36.58M | trainable 36.58M (100%)
~73259 training sequences/epoch | ~2290 steps/epoch (32 seqs/step) | total 9160 steps
VAL loss before training (Phase-1 model, target tokens): 4.4659
  >> speed: 0.54 s/step | projected 20.7 min/epoch (83 min for 4 epochs, plus evaluation) | peak GPU 5.0 GB
  ep 1 step 100/9160 | loss 2.2429 | lr x0.365 | 54s
  ep 1 step 200/9160 | loss 1.2166 | lr x0.730 | 112s
  ep 1 step 300/9160 | loss 0.9937 | lr x1.000 | 170s
  ep 1 step 400/9160 | loss 1.0305 | lr x1.000 | 228s
  ep 1 step 500/9160 | loss 0.9232 | lr x0.999 | 286s
  ep 1 step 600/9160 | loss 0.9507 | lr x0.997 | 345s
  ep 1 step 700/9160 | loss 0.9259 | lr x0.995 | 402s
  ep 1 step 800/9160 |

In [22]:
# Cell 21 — Smoke-test verdict + sample outputs (always printed)
val_extra = {}
_n = min(2, len(val_docs))
_s, _, _parts = summarize_docs(model, val_docs[:_n], return_parts=True)
for d, s in zip(val_docs[:_n], _s):
    print(f"--- {d['id']} | {len(d['chunks'])} chunks | generated {len(s.split())} words | reference {len(d['summ'].split())} words")
    print("GENERATED:", s[:700].replace("\n", " "))
    print("REFERENCE:", d["summ"][:700].replace("\n", " "))
if SMOKE_TEST:
    drop = stats["last_losses"] / stats["first_losses"]
    _ss = summarize_docs(model, val_docs)[0]
    m, _ = rouge_eval(_ss, [d["summ"] for d in val_docs])
    val_extra.update(smoke_r1=m["r1"]["f"], smoke_r1_p=m["r1"]["p"], smoke_r1_r=m["r1"]["r"], smoke_rL=m["rL"]["f"],
                     smoke_words_gen=float(np.mean([len(x.split()) for x in _ss])), smoke_words_ref=float(np.mean([len(d["summ"].split()) for d in val_docs])),
                     smoke_gen_stats=dict(LAST_GEN_STATS))
    print(f"generated {val_extra['smoke_words_gen']:.0f} words vs reference {val_extra['smoke_words_ref']:.0f} | R1 P {m['r1']['p']:.3f} R {m['r1']['r']:.3f} | "
          f"chunks with an empty output {LAST_GEN_STATS['empty_share']:.2f} | tokens/chunk {LAST_GEN_STATS['tokens_per_chunk']:.0f}")
    print(f"\nloss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | VAL loss {history[-1]['val_loss']:.3f} "
          f"(Phase-1 {stats['val_loss_init']:.3f}) | R1 {m['r1']['f']:.3f} RL {m['rL']['f']:.3f} on {len(val_docs)} docs")
    print("SMOKE VERDICT:", "OK -> run the full job" if (drop < 0.95 and history[-1]["val_loss"] < stats["val_loss_init"]) else
          "NOT OK: loss did not fall / VAL loss did not improve -> paste the log to Claude")
    print("Check the '>> speed' line: it projects minutes per epoch; set EPOCHS / TIME_BUDGET_MIN accordingly.")

--- 2459 | 12 chunks | generated 970 words | reference 912 words
GENERATED: The first respondent was a candidate for election to the State Legislative Assembly. The first respondent secured 25,321 votes at the election overpping the votes polled by the second respondent by approximately 1600. The first Respondent filed an Election Petition in the High Court challenging the election of the first respondent. The first appellant contended that he had been duly elected from the constituency and that his election agent had been made to withdraw his nomination paper on payment of an illegal gratification. The first respondent filed a petition for withdrawing the allegations against V.K. Reddi, and in the alternative, to implead him as a party to the election petition. The
REFERENCE: The election of the first respondent to the Andhra Pradesh Legislative Assembly at the General Election hold in 1967 was challenged by the appellants in an election petition. Various corrupt practices were allege

## Decoding sweep and VAL evaluation (full runs)
The decoding configuration is chosen on the first `SWEEP_DOCS` VAL documents by (R1 + RL)/2, then **all 200** VAL documents are summarised with it (the audit's baselines were scored on the same 200). The analysis covers what the PDF asks for: scores vs the extractive baselines, length vs score, redundancy (repeated n-grams), abstractiveness of the outputs (novel n-grams vs the document), scores by document length, and cost.

In [23]:
# Cell 22 — Decoding sweep (on VAL[:SWEEP_DOCS]) -> best GEN
_BS = None
def bertscore_f(cands, refs):
    # roberta-large BERTScore (the competition metric), loaded once; None without a GPU / when disabled
    global _BS
    if not (RUN_BERTSCORE and device.type == "cuda"): return None
    if _BS is None:
        try:
            import bert_score
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True); import bert_score
        import transformers; transformers.logging.set_verbosity_error()
        _BS = bert_score.BERTScorer(model_type=BERTSCORE_MODEL, lang="en", device="cuda", batch_size=8)
    P_, R_, F_ = _BS.score(cands, refs)
    return P_.numpy(), R_.numpy(), F_.numpy()

sweep_res = []
if not SMOKE_TEST and SWEEP_DOCS and DECODE_SWEEP:
    _docs = val_docs[:SWEEP_DOCS]; _refs = [d["summ"] for d in _docs]
    for cfg in DECODE_SWEEP:
        s, secs = summarize_docs(model, _docs, cfg); m, _ = rouge_eval(s, _refs)
        wg = float(np.mean([len(x.split()) for x in s])); bs = bertscore_f(s, _refs)
        r = dict(cfg=cfg, r1=m["r1"]["f"], r2=m["r2"]["f"], rL=m["rL"]["f"], r1_p=m["r1"]["p"], r1_r=m["r1"]["r"], words=wg,
                 empty_chunks=LAST_GEN_STATS["empty_share"], tok_chunk=LAST_GEN_STATS["tokens_per_chunk"], secs=secs)
        if bs is not None: r["bs_f"] = float(bs[2].mean())
        sweep_res.append(r)
        print(f"  {cfg}: R1 {r['r1']:.4f} (P {r['r1_p']:.3f} R {r['r1_r']:.3f}) R2 {r['r2']:.4f} RL {r['rL']:.4f}"
              + (f" | BS {r['bs_f']:.4f}" if "bs_f" in r else "") + f" | words {wg:.0f} | tok/chunk {r['tok_chunk']:.0f} | "
              f"empty {r['empty_chunks']:.2f} | {secs:.0f}s")
    use_bs = SWEEP_SELECT == "bertscore" and all("bs_f" in r for r in sweep_res)
    bestcfg = max(sweep_res, key=lambda r: r["bs_f"] if use_bs else (r["r1"] + r["rL"]) / 2)["cfg"]
    GEN.update(bestcfg); print(f"chosen decoding (by {'BERTScore' if use_bs else '(R1 + RL) / 2'}):", bestcfg)
else:
    print("decoding sweep skipped")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

  {'min_new_all': 64}: R1 0.5891 (P 0.644 R 0.601) R2 0.3289 RL 0.3061 | BS 0.8547 | words 767 | tok/chunk 108 | empty 0.00 | 62s
  {'min_new_all': 96}: R1 0.5972 (P 0.621 R 0.632) R2 0.3343 RL 0.3067 | BS 0.8551 | words 819 | tok/chunk 134 | empty 0.00 | 67s
  {'min_new_all': 128}: R1 0.5948 (P 0.614 R 0.636) R2 0.3310 RL 0.3039 | BS 0.8548 | words 829 | tok/chunk 162 | empty 0.00 | 68s
  {'min_new_all': 32, 'eos_threshold': 0.5}: R1 0.6035 (P 0.633 R 0.628) R2 0.3383 RL 0.3130 | BS 0.8560 | words 801 | tok/chunk 134 | empty 0.00 | 69s
  {'min_new_all': 64, 'eos_threshold': 0.5}: R1 0.6003 (P 0.624 R 0.635) R2 0.3350 RL 0.3082 | BS 0.8557 | words 817 | tok/chunk 149 | empty 0.00 | 69s
  {'min_new_all': 96, 'trim': 'tail'}: R1 0.5957 (P 0.598 R 0.654) R2 0.3330 RL 0.3033 | BS 0.8553 | words 901 | tok/chunk 134 | empty 0.00 | 66s
  {'min_new_all': 96, 'max_chunks': 24}: R1 0.5986 (P 0.622 R 0.634) R2 0.3348 RL 0.3050 | BS 0.8548 | words 825 | tok/chunk 134 | empty 0.00 | 74s
  {'min_new

In [24]:
# Cell 23 — VAL evaluation: ROUGE (+ BERTScore), lengths, redundancy, abstractiveness, by document length
val_res = {}
if not SMOKE_TEST:
    refs = [d["summ"] for d in val_docs]
    val_sum, t_val = summarize_docs(model, val_docs)
    val_extra["gen_stats"] = dict(LAST_GEN_STATS)
    m, per = rouge_eval(val_sum, refs)
    val_res = {k: v["f"] for k, v in m.items()}
    val_extra.update(seconds=t_val, sec_per_doc=t_val / len(val_docs),
                     words_gen=float(np.mean([len(s.split()) for s in val_sum])), words_ref=float(np.mean([len(r.split()) for r in refs])),
                     r1_p=m["r1"]["p"], r1_r=m["r1"]["r"])
    _b = bertscore_f(val_sum, refs)
    if _b is not None:
        val_res.update(bs_p=float(_b[0].mean()), bs_r=float(_b[1].mean()), bs_f=float(_b[2].mean())); per["bs"] = _b[2]
    print(f"===== VAL ({len(val_docs)} docs) | R1 {val_res['r1']:.4f} (P {m['r1']['p']:.3f} R {m['r1']['r']:.3f}) | R2 {val_res['r2']:.4f} | "
          f"RL {val_res['rL']:.4f}" + (f" | BERTScore F1 {val_res['bs_f']:.4f}" if "bs_f" in val_res else ""))
    for name, b in BASELINES.items():
        print(f"      vs {name:6s}: R1 {val_res['r1'] - b['r1']:+.4f} | R2 {val_res['r2'] - b['r2']:+.4f} | RL {val_res['rL'] - b['rL']:+.4f}"
              + (f" | BERTScore {val_res['bs_f'] - b['bs']:+.4f}" if "bs_f" in val_res else ""))
    print(f"      words: generated {val_extra['words_gen']:.0f} vs reference {val_extra['words_ref']:.0f} | {val_extra['sec_per_doc']:.1f} s/doc")

    # redundancy: share of repeated 3-grams inside each output; abstractiveness: novel n-grams w.r.t. the document
    def rep3(t):
        w = words(t); g = list(zip(w, w[1:], w[2:]))
        return 1 - len(set(g)) / max(len(g), 1)
    def novel(t, d, n):
        a, s = words(d), words(t); ga = set(zip(*[a[i:] for i in range(n)])); gs = list(zip(*[s[i:] for i in range(n)]))
        return sum(x not in ga for x in gs) / max(len(gs), 1)
    val_extra["rep3_gen"] = float(np.mean([rep3(s) for s in val_sum])); val_extra["rep3_ref"] = float(np.mean([rep3(r) for r in refs]))
    val_extra["novel"] = {n: float(np.mean([novel(s, d["doc"], n) for s, d in zip(val_sum, val_docs)])) for n in (1, 2, 4)}
    print(f"      redundancy (repeated 3-grams): generated {val_extra['rep3_gen']:.3f} vs reference {val_extra['rep3_ref']:.3f}")
    print(f"      novel n-grams vs the document (1/2/4): {[round(v, 3) for v in val_extra['novel'].values()]}  (references: 0.044 / 0.223 / 0.484)")

    ntok_v = np.asarray([d["ntok"] for d in val_docs]); cuts = np.percentile(ntok_v, [33.3, 66.7]); bk = np.digitize(ntok_v, cuts); rows = []
    print("      by document length (token terciles):")
    for b_, name in enumerate(("short", "medium", "long")):
        sel = bk == b_
        r_ = dict(bucket=name, n=int(sel.sum()), median_tokens=float(np.median(ntok_v[sel])), r1=float(per["r1"][sel].mean()),
                  rL=float(per["rL"][sel].mean()), truncated=float(np.mean([len(val_docs[i]["chunks"]) > GEN["max_chunks"] for i in np.where(sel)[0]])))
        if "bs" in per: r_["bs"] = float(per["bs"][sel].mean())
        rows.append(r_)
        print(f"        {name:6s}: {sel.sum():3d} docs | median {r_['median_tokens']:6.0f} tokens | R1 {r_['r1']:.3f} RL {r_['rL']:.3f}"
              + (f" BS {r_['bs']:.3f}" if "bs" in r_ else "") + f" | chunks skipped in {100 * r_['truncated']:.0f}% of docs")
    val_extra["by_length"] = rows
    (RUN_DIR / "val_summaries.json").write_text(json.dumps([dict(id=d["id"], summary=s) for d, s in zip(val_docs, val_sum)], ensure_ascii=False))
    TM = TEACHER["meta"]
    if TM.get("val"):
        tv_ = TM["val"]
        print(f"      TEACHER {TM['teacher']} ({TM['params_total']/1e6:.1f}M): R1 {tv_.get('r1', 0):.4f} R2 {tv_.get('r2', 0):.4f} RL {tv_.get('rL', 0):.4f} "
              f"BERTScore {tv_.get('bs_f', 0):.4f}  vs STUDENT ({model.n_total/1e6:.1f}M, {100*model.n_total/TM['params_total']:.0f}% of the teacher)")
        val_extra["teacher_val"] = tv_; val_extra["teacher_params"] = TM["params_total"]
else:
    print("no VAL evaluation (smoke run)")

===== VAL (200 docs) | R1 0.5946 (P 0.624 R 0.628) | R2 0.3336 | RL 0.3109 | BERTScore F1 0.8544
      vs tfidf : R1 +0.0197 | R2 +0.0150 | RL +0.0101 | BERTScore +0.0011
      vs lead  : R1 +0.0724 | R2 +0.0828 | RL +0.0672 | BERTScore +0.0223
      vs oracle: R1 -0.1914 | R2 -0.2706 | RL -0.2232 | BERTScore -0.0311
      words: generated 788 vs reference 848 | 0.7 s/doc
      redundancy (repeated 3-grams): generated 0.100 vs reference 0.099
      novel n-grams vs the document (1/2/4): [0.029, 0.132, 0.318]  (references: 0.044 / 0.223 / 0.484)
      by document length (token terciles):
        short :  67 docs | median   2033 tokens | R1 0.595 RL 0.347 BS 0.863 | chunks skipped in 0% of docs
        medium:  66 docs | median   4260 tokens | R1 0.591 RL 0.309 BS 0.854 | chunks skipped in 0% of docs
        long  :  67 docs | median   8362 tokens | R1 0.598 RL 0.277 BS 0.846 | chunks skipped in 54% of docs
      TEACHER p6_encdec_adapter (63.5M): R1 0.5956 R2 0.3355 RL 0.3092 BERTScore 

In [25]:
# Cell 24 — Test summaries -> prediction.json (list format) -> submission.zip (full runs only)
def verify_submission(zip_path, docs):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list), "prediction.json must be a JSON list of {id, summary}"
    want = [str(d["id"]) for d in docs]; got = [str(e["id"]) for e in sub]
    assert len(got) == len(set(got)), "duplicate ids"
    assert set(got) == set(want), f"id mismatch: missing {len(set(want) - set(got))}, extra {len(set(got) - set(want))}"
    for e in sub:
        assert set(e) == {"id", "summary"}, f"unexpected keys {set(e)}"
        assert isinstance(e["summary"], str) and e["summary"].strip(), f"{e['id']}: empty summary"
    print(f"verified: {len(sub)} summaries, one non-empty string per test id")

if not SMOKE_TEST:
    prepare_docs(TEST_ALL, "test", False)
    test_sum, t_test = summarize_docs(model, TEST_ALL)
    sub = [dict(id=d["id"], summary=s) for d, s in zip(TEST_ALL, test_sum)]          # id keeps its type from test.json
    out_dir = RUN_DIR / "submission"; out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    verify_submission(out_dir / "submission.zip", TEST_ALL)
    print(f"test: {len(sub)} docs in {t_test:.0f}s | words/summary mean {np.mean([len(s.split()) for s in test_sum]):.0f}")
else:
    print("smoke run: no submission written")

prepare test:   0%|          | 0/100 [00:00<?, ?it/s]

verified: 100 summaries, one non-empty string per test id
test: 100 docs in 68s | words/summary mean 787


In [26]:
# Cell 25 — Save trainable weights + run log
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, Path): return str(o)
    return o
log = dict(run=RUN_NAME, method=METHOD, hp=HP[METHOD], chunk_tokens=CHUNK_TOKENS, tgt_max=TGT_MAX, empty_keep=EMPTY_KEEP,
           min_align_f1=MIN_ALIGN_F1, epochs=EPOCHS, micro_tokens=MICRO_TOKENS, step_seqs=STEP_SEQS, select_by=SELECT_BY,
           gen=GEN, decode_sweep=sweep_res, params_total=model.n_total, params_trainable=model.n_trainable,
           history=history, best_epoch=best["epoch"], stats=stats, val=val_res, val_extra=val_extra, baselines=BASELINES,
           n_train_docs=len(train_docs), n_val_docs=len(val_docs), smoke=SMOKE_TEST, kd_alpha=KD_ALPHA, seq_kd=SEQ_KD, student_layers=STUDENT_KEEP_LAYERS, kd_stats=dict(KD_STATS), teacher=TEACHER['meta'].get('teacher'))
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "hp": HP[METHOD], "gen": GEN, "log": _jsonable(log)}, RUN_DIR / f"summ_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['best_trainable.pt', 'run_log.json', 'submission', 'summ_p6_kd_dec03_student.pt', 'val_summaries.json']


In [27]:
# Cell 27 — PASTE TO CLAUDE
short = dict(run=RUN_NAME, smoke=SMOKE_TEST, hp=HP[METHOD], kd_alpha=KD_ALPHA, seq_kd=SEQ_KD, kd_stats=dict(KD_STATS), params_trainable=model.n_trainable, params_total=model.n_total,
             history=history, best_epoch=best["epoch"], stats={k: (round(v, 4) if isinstance(v, float) else v) for k, v in stats.items()},
             gen=GEN, decode_sweep=sweep_res, val={k: round(v, 4) for k, v in val_res.items()},
             val_extra={k: v for k, v in val_extra.items() if k != "seconds"})
print("================ PASTE TO CLAUDE ================")
print(json.dumps(_jsonable(short), separators=(",", ":")))
print("=================================================")

================ PASTE TO CLAUDE ================
{"run":"p6_kd_dec03_student","smoke":false,"hp":{"lr":0.0003,"wd":0.0},"kd_alpha":0.5,"seq_kd":true,"kd_stats":{"with_logits":185900,"length_mismatch":2458},"params_trainable":36577664,"params_total":36577664,"history":[{"epoch":1,"train_loss":1.1904640127107358,"val_loss":1.6956216609598325,"seconds":1072.6610627174377,"steps":1808,"q_r1":0.5698555678311112,"q_r2":0.29834417759376025,"q_rL":0.28828516142711524},{"epoch":2,"train_loss":0.8927991063723051,"val_loss":1.5621374902164833,"seconds":1078.0128264427185,"steps":3616,"q_r1":0.5887938022281091,"q_r2":0.314412954515377,"q_rL":0.30074589543061747},{"epoch":3,"train_loss":0.7768852957048477,"val_loss":1.5078334117334242,"seconds":1078.935749053955,"steps":5421,"q_r1":0.5907055251776183,"q_r2":0.3180498388963281,"q_rL":0.30226115388010316},{"epoch":4,"train_loss":0.6849991808868058,"val_loss":1.4790237592940074,"seconds":1081.4517407417297,"steps":7227,"q_r1":0.5951846852796047,"q_r2